In [1]:
import osmnx as ox

ox._version.__version__

'1.9.4'

In [3]:
def load_G_simplyfied():
    g = ox.load_graphml("tests/data/test_rng.ml")
    g = ox.simplify_graph(g)
    return g

G = load_G_simplyfied()

In [7]:
from genesis.best_points import BestNodeHillClimbing
from genesis.lscp import LSCPCommon
from genesis.mclp import BestNodesKoptG
from genesis.metrics import ArrivalTime
from genesis.point_selectors import GenesisNodeSelector
from genesis.states import FirstArrivalUnitState
from genesis.stop_cases import ItersStopCase




nodes = list(G.nodes())
existed_units = {nodes[100]: 'A',
        nodes[200]: 'B',
        }
new_units = {nodes[300]: 'c',
            nodes[400]: 'd'}

# Функция расчета лучшего узла в графе
BNHC = BestNodeHillClimbing(state_function = FirstArrivalUnitState(),
                            metric_function = ArrivalTime())
# Функция расчета лучшего размещения заданных узлов
BNG = BestNodesKoptG(state_function = FirstArrivalUnitState(),
                    metric_function = ArrivalTime(),
                    best_point_function = BNHC,
                    iterations = 5,
                    )
# Критерий остановки
stop_case = ItersStopCase(max_iter_count = 2)
# Селектор следующей точки
point_selector = GenesisNodeSelector(state_function = FirstArrivalUnitState(),
                                    metric_function = ArrivalTime())


# Сборка алгоритма
genesis = LSCPCommon(mclp_function = BNG,
                    point_selector = point_selector,
                    stop_case_function = stop_case,
                    metric_function = ArrivalTime(),
                    start_names_index = 5,
                    names_pattern='ПСЧ {}',
                    )

# Расчет только динамические узлы
# best_dynamic_nodes, best_metric = genesis(env = G,
#                                 dynamic_nodes = new_units,
#                                 )

# # assert best_dynamic_nodes == {290700367: 'd', 1577701962: 'c', 2034402108: 'ПСЧ 5', 2042076918: 'ПСЧ 6'}
# assert best_dynamic_nodes == {290700367: 'd', 1577701962: 'c'}
# assert best_metric  ==  4.5251939000000005

# Расчет динамические и статические узлы
best_dynamic_nodes, best_metric = genesis(env = G,
                                dynamic_nodes = new_units,
                                static_nodes = existed_units,
                                )
# assert best_dynamic_nodes == {2034401374: 'c', 290700367: 'd', 2034402108: 'ПСЧ 5', 2042076918: 'ПСЧ 6'}
# assert best_metric  ==  4.3977074946843855
best_dynamic_nodes, best_metric

({2034401275: 'c', 2042076918: 'd', 290700365: 'ПСЧ 5', 2034401894: 'ПСЧ 6'},
 4.392374431672204)

In [ ]:
({1577701962: 'c', 2042076918: 'd', 290700365: 'ПСЧ 5', 2034401723: 'ПСЧ 6'},
 4.47569295393134)

In [9]:
import geopandas as gpd
def load_area():
    return gpd.read_file("tests/data/test_polygon.gpkg")

area = load_area()

In [10]:
nodes = list(G.nodes())
existed_units = {nodes[100]: 'A',
        nodes[200]: 'B',
        }
new_units = {nodes[300]: 'c',
            nodes[400]: 'd'}

g_nodes_gdf = ox.graph_to_gdfs(G, edges=False)
points_mask = g_nodes_gdf.within(area.iloc[0].geometry)

# Функция расчета лучшего узла в графе
BNHC = BestNodeHillClimbing(state_function = FirstArrivalUnitState(),
                            metric_function = ArrivalTime())
# Функция расчета лучшего размещения заданных узлов
BNG = BestNodesKoptG(state_function = FirstArrivalUnitState(),
                    metric_function = ArrivalTime(),
                    best_point_function = BNHC,
                    iterations = 5,
                    )
# Критерий остановки
stop_case = ItersStopCase(max_iter_count = 2)
# Селектор следующей точки
point_selector = GenesisNodeSelector(state_function = FirstArrivalUnitState(),
                                    metric_function = ArrivalTime())

# Сборка алгоритма
genesis = LSCPCommon(mclp_function = BNG,
                    point_selector = point_selector,
                    stop_case_function = stop_case,
                    metric_function = ArrivalTime(),
                    start_names_index = 10,
                    names_pattern='ПЧ {}',
                    )

# Расчет только динамические узлы
best_dynamic_nodes, best_metric = genesis(env = G,
                                dynamic_nodes = new_units,
                                static_nodes = existed_units,
                                area = points_mask
                                )

best_dynamic_nodes, best_metric

({2034401275: 'c', 9774067520: 'd', 2034401650: 'ПЧ 10', 426923846: 'ПЧ 11'},
 2.5924815766972755)